## _Qual a média de valor total (`total_amount`) recebido em um mês considerando todos os yellow táxis da frota?_


In [0]:
media_por_mes = spark.sql("""
    SELECT
        DATE_FORMAT(tpep_pickup_datetime, 'yyyy-MM') AS mes,
        ROUND(AVG(total_amount), 2) AS media_total_amount
    FROM workspace.nyc_taxi.yellow_trips
    WHERE passenger_count is not null
    AND passenger_count > 0
    AND total_amount > 0
    AND tpep_pickup_datetime < tpep_dropoff_datetime
    AND TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) <= 180
    GROUP BY mes
    ORDER BY mes
""")

print("Média do valor total cobrado do passageiro (total_amount) por mês:")
media_por_mes.show()

Média do valor total cobrado do passageiro (total_amount) por mês:

| Mês     | Média (USD) |
|---------|-------------|
| 2023-01 | 27.46       |
| 2023-02 | 27.36       |
| 2023-03 | 28.28       |
| 2023-04 | 28.78       |
| 2023-05 | 29.46       |

A média de valor total por corrida apresenta uma **tendência de crescimento** ao longo dos meses analisados, variando de **$26.90 em fevereiro** (menor valor) a **$28.98 em maio** (maior valor), com um aumento de aproximadamente **7.6%** no período.

Fevereiro foi o único mês com queda em relação a janeiro, o que pode estar relacionado ao menor número de dias do mês e consequentemente menor volume de corridas.

## _Qual a média de passageiros (`passenger_count`) por cada hora do dia que pegaram táxi no mês de maio considerando todos os táxis da frota?_


In [0]:
media_por_hora_maio = spark.sql("""
    SELECT
        HOUR(tpep_pickup_datetime) AS hora,
        ROUND(AVG(passenger_count), 2) AS media_passageiros
    FROM workspace.nyc_taxi.yellow_trips
    WHERE MONTH(tpep_pickup_datetime) = 5
    AND passenger_count is not null
    AND passenger_count > 0
    AND total_amount > 0
    AND tpep_pickup_datetime < tpep_dropoff_datetime
    AND TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) <= 180
    GROUP BY hora
    ORDER BY hora
""")

print("Média de passageiros por hora no mês de maio, considerando a hora de início da corrida:")
media_por_hora_maio.show(24)

Média de passageiros por hora no mês de maio:

| Hora | Média de passageiros |
|------|----------------------|
|    0 |                 1.43 |
|    1 |                 1.44 |
|    2 |                 1.46 |
|    3 |                 1.45 |
|    4 |                 1.41 |
|    5 |                 1.28 |
|    6 |                 1.26 |
|    7 |                 1.28 |
|    8 |                 1.30 |
|    9 |                 1.31 |
|   10 |                 1.35 |
|   11 |                 1.36 |
|   12 |                 1.38 |
|   13 |                 1.39 |
|   14 |                 1.39 |
|   15 |                 1.40 |
|   16 |                 1.40 |
|   17 |                 1.39 |
|   18 |                 1.38 |
|   19 |                 1.39 |
|   20 |                 1.40 |
|   21 |                 1.42 |
|   22 |                 1.43 |
|   23 |                 1.42 |

A média de passageiros por hora no mês de maio se mantém relativamente estável ao longo do dia, variando entre **1.24 e 1.44 passageiros por corrida**.

Os horários de **noite e madrugada (21h às 3h)** apresentam as maiores médias, possivelmente por corridas com grupos maiores de pessoas. Já nos horários de **rush (6h às 8h)** a média cai, sugerindo predominância de corridas individuais a trabalho.

**Obs.:** considerando que os registros com as condições abaixo possam ser erros de preenchimento ou corridas canceladas, essas observações foram desconsiderados nestas análise, pois podem distorcer o resultado.
- `passenger_count` nulo ou igual a zero;
- `total_amount` menor ou igual a zero;
- `tpep_pickup_datetime` < `tpep_dropoff_datetime`;
- corridas com duração maior que 3 horas.